In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [5]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 2000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [7]:
messages = []

add_user_message(messages, thinking_test_str)

chat(messages, thinking=True)

Message(id='msg_011CgxvSJTQQT5iVozyAw4qr', container=None, content=[RedactedThinkingBlock(data='EsUECroBCBMYAipAdJlVdOD9uBTliNJ2trUW+fQ+F3g93XoPwAS29uUTApjmOVyhIMCWHkOlogJU6JeKVRiH+LLp0jWfXs/YZY0xPDIaY2xhdWRlLXNvbm5ldC00LTUtMjAyNTA5Mjk4AEIIdGhpbmtpbmdaJDFhMTFmN2RhLTI4NmYtNDRmYi1iZWVkLTdiMDdmYjFhY2M3NpoBHAoaY2xhdWRlLXNvbm5ldC00LTUtMjAyNTA5MjmoAfm51tcGEgyNRJZNAHWGQIqDldIaDCos0HPSKWnTB7O8BCIwRFo1d8s5zq+1/1g4bcH7p6/cSKOO7V3h7+FoBUUcN6lTuwtSGiyu0vrJ4a56FT6sKrcC96EDm/Sb7bnZAHBwiU3rgCp2H3zSUHS6fi1xLCAKsUVmH1ea/2D2TxNyjDtd9tDfvj8fm+LETmYa8ObPFp6Hb2AunFew20q9w/ExzDEhrGdfccdvVCH9xjsbhA9AFYq/MoF/sY8KQK6m+HAOfaS3WdYPR+hct4t0VLvtsV0asiMghd8X8nTYSM5lZBM50EtopAeH0GqVTtlLAEAF4lrgNKpYjUUfSeuoSo+YbdO0fCNgUHIZokbGAM8BosoW/aK0xg8qwf/75ks2qNZp7Ui7FPfhGqHMDmMiBk/Xxu0LJs6xEFxeUV4DO2xHKopvPy5FO7wpz6ksWL2K/6aV/vvC4nSMI7rkOQ+zd2hEw9peq0TdFp6dRXTAWhey+KWIy74JRbAohDxTFhNiD/7MYeza2h9ZuMiC3N8YAQ==', type='redacted_thinking'), TextBlock(citations=None, text="I'm here to help you! It looks like you've sent a test str

In [9]:
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [10]:
# TODO: Read image data, feed into Claude
import base64

with open("./images/prop7.png","rb") as f:
        image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "What do you see in this image?"
    }
])



In [11]:
chat(messages)

Message(id='msg_011CgxvXtn3UsUsPD9Rnnoo8', container=None, content=[TextBlock(citations=None, text="# Analysis\n\nThis appears to be an aerial/satellite view image showing:\n\n- **A house or building structure** in the center with what looks like a gray/light-colored roof\n- **Dense forest or tree canopy** surrounding the structure on all sides, showing various shades of green\n- **Heavy tree coverage** creating a secluded, wooded setting\n- The building appears to have an **irregular or L-shaped footprint**\n- What might be a **driveway or path** visible on the left side of the image\n- The image quality suggests it's taken from directly above, possibly from satellite imagery or drone footage\n\nThe property appears quite isolated and well-concealed by the surrounding vegetation, with the trees creating a natural buffer around the building.", type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='messag

In [25]:
# TODO: Read image data, feed into Claude
import base64
messages = []

with open("./earth.pdf","rb") as f:
        file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title":"earth.pdf",
        "citations":{"enabled":True}
    },
    # Text Block
    {
        "type": "text",
        "text": "How earth is formed"
    }
])



In [26]:
chat(messages)

Message(id='msg_011Cgy9dBwKBQcKZ5aJZjeQR', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text='Earth, like most other bodies in the Solar System,\r\nformed about 4.5\xa0billion years ago from gas and dust in\r\nthe early Solar System. ', document_index=0, document_title='earth.pdf', end_page_number=3, file_id=None, start_page_number=2, type='page_location')], text='Earth, like most other bodies in the Solar System, formed about 4.5 billion years ago from gas and dust in the early Solar System.', type='text'), TextBlock(citations=None, text=' More specifically, ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='In theory, a solar nebula\r\npartitions a volume out of a molecular cloud by\r\ngravitational collapse, which begins to spin and\r\nflatten into a circumstellar disk, and then the\r\nplanets grow out of that disk with the Sun. ', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type=

In [27]:
article_text = """

The oldest material found in the Solar System is
dated to 4.5682 +0.0002−0.0004 Ga (billion years) ago.[35]
By  4.54 ±0.04  Ga  the  primordial  Earth  had
formed.[36] The bodies in the Solar System formedand evolved with the Sun. In theory, a solar nebulapartitions  a  volume  out  of  a  molecular  cloud  by
gravitational  collapse,  which  begins  to  spin  and
flatten  into  a  circumstellar  disk,  and  then  the
planets  grow  out  of  that  disk  with  the  Sun.  A
nebula contains gas, ice grains, and dust (includingprimordial nuclides). According to nebular theory,planetesimals  formed  by  accretion,  with  the
primordial Earth being estimated as likely taking
anywhere from 70 to 100 million years to form.[37]
Estimates of the age of the Moon range from 4.5
Ga to significantly younger.[38] A leading hypothesis is that it was formed by accretion from material
loosed from Earth after a Mars-sized object with about 10% of Earth's mass, named Theia, collided
with Earth.[39] It hit Earth with a glancing blow and some of its mass merged with Earth.[40][41]
Between  approximately  4.0  and  3.8  Ga,  numerous  asteroid  impacts  during  the  Late  HeavyBombardment caused significant changes to the greater surface environment of the Moon and, by
inference, to that of Earth



"""

In [34]:

messages = []
add_user_message(messages, 
[
{
    "type": "document", 
    "source": {
        "type": "text",
        "media_type": "text/plain",
        "data": article_text,
    },

    "title": "earth_article",
    "citations": { "enabled": True }
},

{
        "type": "text",
        "text": "Summarize the document and cite?"
}

])
chat(messages)



Message(id='msg_011CgyAXf1C9V7SHL1nJF8im', container=None, content=[TextBlock(citations=None, text='This document describes the formation and early history of Earth and the Solar System. ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='\n\nThe oldest material found in the Solar System is\ndated to 4.5682 +0.0002−0.0004 Ga (billion years) ago.', document_index=0, document_title='earth_article', end_char_index=105, file_id=None, start_char_index=0, type='char_location')], text='The oldest material found in the Solar System is dated to 4.5682 billion years ago.', type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='[35]\nBy  4.54 ±0.04  Ga  the  primordial  Earth  had\nformed.', document_index=0, document_title='earth_article', end_char_index=165, file_id=None, start_char_index=105, type='char_location')], text='The primordial Earth had formed by 4.54 billion years ago.', type='text'), TextBlock(citatio